In [25]:
import os
import json
import math
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 判斷 GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [26]:
# -----------------------------
# Step 1. 讀取資料 & 處理負值
# -----------------------------
def load_and_prepare_data(file_paths):
    """
    載入多個 CSV，合併後只保留:
      DateTime, Tx, RH, PrecpHour
    將 <0 視為缺失(NaN)，再用 'nearest' 插值補齊。
    """
    all_dfs = []
    for fp in file_paths:
        df = pd.read_csv(fp, parse_dates=["DateTime"])
        df = df[["DateTime", "Tx", "RH", "PrecpHour"]]
        all_dfs.append(df)
    
    # 合併 & 排序
    full_df = pd.concat(all_dfs, ignore_index=True)
    full_df.sort_values(by="DateTime", inplace=True)
    full_df.reset_index(drop=True, inplace=True)
    
    # 將負值視為缺失
    for col in ["Tx", "RH", "PrecpHour"]:
        full_df[col] = full_df[col].mask(full_df[col] < 0, np.nan)
    
    # 用最鄰近插值補齊
    full_df[["Tx","RH","PrecpHour"]] = (
        full_df[["Tx","RH","PrecpHour"]]
        .interpolate(method="nearest", limit_direction="both")
    )
    
    return full_df

# 讀取訓練資料 (2020 ~ 2022)
train_df = load_and_prepare_data([
    "Data/466920_2020.csv",
    "Data/466920_2021.csv",
    "Data/466920_2022.csv"
])

# 讀取「驗證」用的資料 (2023 + 2024)，
# 因為驗證要輸出 2024，若週數太大，得回溯到 2023
val_df_raw = load_and_prepare_data([
    "Data/466920_2023.csv",
    "Data/466920_2024.csv"
])

In [27]:
# -----------------------------
# Step 2. 設計「訓練」用 Dataset
#     (滑動窗口方式)
# -----------------------------
class TrainDataset(Dataset):
    """
    用於訓練 (滑動窗口):
      X: (num_history_weeks * 168) 筆
      Y: 168 筆 (下一週)
    """
    def __init__(self, df, num_history_weeks=1):
        super().__init__()
        self.num_history_hours = num_history_weeks * 168
        self.predict_hours = 168  # 固定預測一週
        
        self.features = df[["Tx","RH","PrecpHour"]].values.astype(np.float32)
        self.datetimes = df["DateTime"].values
        
    def __len__(self):
        return len(self.features) - (self.num_history_hours + self.predict_hours) + 1
    
    def __getitem__(self, idx):
        x_start = idx
        x_end   = idx + self.num_history_hours
        y_start = x_end
        y_end   = x_end + self.predict_hours
        
        X = self.features[x_start:x_end]  # (num_history_hours, 3)
        Y = self.features[y_start:y_end]  # (168, 3)
        return X, Y

def create_train_loader(train_df, num_history_weeks, batch_size=16, shuffle=True):
    dataset = TrainDataset(train_df, num_history_weeks=num_history_weeks)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)
    return loader

In [28]:
# -----------------------------
# Step 3. 設計「驗證」用 Dataset
#     (只輸出 2024 範圍，但可回溯 2023)
# -----------------------------
class ValDataset(Dataset):
    """
    為了驗證 2024 年的預測週數，
    但可能回朔到 2023 的資料作為 X。
    
    只保證 Y (要預測的一週) 落在 [start_date, end_date) 之間，
    但 X 可動態回溯 num_history_weeks。
    """
    def __init__(self, df, num_history_weeks=1,
                 start_date="2024-01-01", end_date="2024-12-31"):
        super().__init__()
        self.num_history_hours = num_history_weeks * 168
        self.predict_hours = 168
        
        self.df = df  # 2023 + 2024
        self.full_features = df[["Tx","RH","PrecpHour"]].values.astype(np.float32)
        self.full_dates = df["DateTime"].values
        
        # 篩出要當作 "Y" 的區間 (在 2024 年內)
        self.df_filtered = df[(df["DateTime"] >= start_date) & (df["DateTime"] < end_date)].copy()
        self.filt_features = self.df_filtered[["Tx","RH","PrecpHour"]].values
        self.filt_dates = self.df_filtered["DateTime"].values
        
    def __len__(self):
        # 能形成多少 "Y" 區段 (一週)
        return len(self.filt_features) - self.predict_hours + 1
    
    def __getitem__(self, idx):
        # 1) 找到 Y 區段 (在過濾後的 df_filtered 中)
        y_start_time = self.filt_dates[idx]
        y_end_time   = y_start_time + pd.Timedelta(hours=self.predict_hours)
        
        mask_y = (self.df["DateTime"] >= y_start_time) & (self.df["DateTime"] < y_end_time)
        Y_df = self.df.loc[mask_y, ["Tx","RH","PrecpHour"]]
        
        if len(Y_df) < 168:
            raise ValueError(f"Y 資料不足 {len(Y_df)}/168, 時間 {y_start_time}~{y_end_time}")
        
        # 2) 找到 X 區段 => 往前 num_history_hours
        x_end_time = y_start_time
        x_start_time = x_end_time - pd.Timedelta(hours=self.num_history_hours)
        
        mask_x = (self.df["DateTime"] >= x_start_time) & (self.df["DateTime"] < x_end_time)
        X_df = self.df.loc[mask_x, ["Tx","RH","PrecpHour"]]
        
        if len(X_df) < self.num_history_hours:
            raise ValueError(f"X 資料不足 {len(X_df)}/{self.num_history_hours}, 回溯時間: {x_start_time}~{x_end_time}")
        
        X = X_df.values.astype(np.float32)
        Y = Y_df.values[:168].astype(np.float32)
        
        return X, Y

def create_val_loader(val_df, num_history_weeks, 
                      start_date="2024-01-01", end_date="2024-12-20",
                      batch_size=1):
    dataset = ValDataset(val_df, num_history_weeks=num_history_weeks,
                         start_date=start_date, end_date=end_date)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    return loader


In [29]:
# -----------------------------
# Step 4. 建立 Transformer 模型
# -----------------------------
class PositionalEncoding(nn.Module):
    """
    基本位置編碼
    """
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        
        self.register_buffer('pe', pe.unsqueeze(0))  # (1, max_len, d_model)
        
    def forward(self, x):
        seq_len = x.size(1)
        return x + self.pe[:, :seq_len, :].to(x.device)

class TransformerModel(nn.Module):
    def __init__(
        self,
        input_dim=3,       # (Tx, RH, PrecpHour)
        d_model=32,
        nhead=4,
        num_encoder_layers=2,
        num_decoder_layers=2,
        dim_feedforward=64,
        output_dim=3       # (Tx, RH, PrecpHour)
    ):
        super().__init__()
        
        self.input_fc = nn.Linear(input_dim, d_model)
        self.pos_encoder = PositionalEncoding(d_model)
        
        self.transformer = nn.Transformer(
            d_model=d_model,
            nhead=nhead,
            num_encoder_layers=num_encoder_layers,
            num_decoder_layers=num_decoder_layers,
            dim_feedforward=dim_feedforward,
            batch_first=True
        )
        self.output_fc = nn.Linear(d_model, output_dim)
        
    def forward(self, src, tgt):
        """
        src: (batch, src_len, 3)
        tgt: (batch, 168, 3)
        回傳: (batch, 168, 3)
        """
        src_emb = self.input_fc(src)
        tgt_emb = self.input_fc(tgt)
        
        src_emb = self.pos_encoder(src_emb)
        tgt_emb = self.pos_encoder(tgt_emb)
        
        out = self.transformer(src_emb, tgt_emb)
        out = self.output_fc(out)
        return out

In [30]:
# -----------------------------
# Step 5. 訓練 & 驗證
# -----------------------------
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0
    for X, Y in loader:
        X = X.to(device)
        Y = Y.to(device)
        
        optimizer.zero_grad()
        outputs = model(X, Y)  # Teacher forcing
        loss = criterion(outputs, Y)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    return total_loss / len(loader)

def evaluate_val(model, loader):
    """
    在驗證集上計算:
    1) 全部三特徵合併後的 MAE, MSE, RMSE, R²
    2) Tx 的 MAE, MSE, RMSE, R²
    3) RH 的 MAE, MSE, RMSE, R²
    4) Precp 是否下雨 (0/1) 的 Accuracy
    """
    model.eval()
    all_preds = []
    all_trues = []
    
    with torch.no_grad():
        for X, Y in loader:
            X = X.to(device)
            Y = Y.to(device)
            preds = model(X, Y)  # Teacher forcing
            
            all_preds.append(preds.cpu().numpy())
            all_trues.append(Y.cpu().numpy())
    
    # 合併 batch 維度 => [N, 168, 3]
    all_preds = np.concatenate(all_preds, axis=0)
    all_trues = np.concatenate(all_trues, axis=0)
    
    # flatten => [N*168, 3]
    all_preds_flat = all_preds.reshape(-1, 3)
    all_trues_flat = all_trues.reshape(-1, 3)
    
    # ========== (A) 整體三特徵計算 ==========
    mae_all = mean_absolute_error(all_trues_flat, all_preds_flat)
    mse_all = mean_squared_error(all_trues_flat, all_preds_flat)
    rmse_all = math.sqrt(mse_all)
    r2_all = r2_score(all_trues_flat, all_preds_flat)
    
    # ========== (B) 分開取出 Tx, RH, Precp ==========
    # 0 -> Tx, 1 -> RH, 2 -> PrecpHour
    true_tx = all_trues_flat[:, 0]
    pred_tx = all_preds_flat[:, 0]
    
    true_rh = all_trues_flat[:, 1]
    pred_rh = all_preds_flat[:, 1]
    
    true_precp = all_trues_flat[:, 2]
    pred_precp = all_preds_flat[:, 2]
    
    # ========== (B1) Tx ==========
    mae_tx = mean_absolute_error(true_tx, pred_tx)
    mse_tx = mean_squared_error(true_tx, pred_tx)
    rmse_tx = math.sqrt(mse_tx)
    r2_tx = r2_score(true_tx, pred_tx)
    
    # ========== (B2) RH ==========
    mae_rh = mean_absolute_error(true_rh, pred_rh)
    mse_rh = mean_squared_error(true_rh, pred_rh)
    rmse_rh = math.sqrt(mse_rh)
    r2_rh = r2_score(true_rh, pred_rh)
    
    # ========== (B3) Precp Accuracy (是否下雨) ==========
    # 將預測值與真實值二值化：>0 => 1；=0 => 0
    pred_precp_bin = (pred_precp > 0).astype(int)
    true_precp_bin = (true_precp > 0).astype(int)
    acc_precp = (pred_precp_bin == true_precp_bin).mean()
    
    # 回傳多項指標，可用 dict 集中管理
    return {
        "mae_all": mae_all,
        "mse_all": mse_all,
        "rmse_all": rmse_all,
        "r2_all": r2_all,
        
        "mae_tx": mae_tx,
        "mse_tx": mse_tx,
        "rmse_tx": rmse_tx,
        "r2_tx": r2_tx,
        
        "mae_rh": mae_rh,
        "mse_rh": mse_rh,
        "rmse_rh": rmse_rh,
        "r2_rh": r2_rh,
        
        "acc_precp": acc_precp
    }



In [31]:
# -----------------------------
# Step 6. 實驗流程 & 寫入 log.json
# -----------------------------
def run_experiment_and_log(
    train_df,        # 2020~2022
    val_df,          # 2023+2024
    num_history_weeks_list,
    epochs=5,
    log_path="log.json"
):
    # 若已存在 log，先載入
    if os.path.exists(log_path):
        with open(log_path, "r") as f:
            logs = json.load(f)
    else:
        logs = []
    
    for n_hw in num_history_weeks_list:
        print(f"\n=== Start Experiment: num_history_weeks={n_hw} ===")
        
        # 建立 train_loader
        train_loader = create_train_loader(train_df, n_hw, batch_size=16, shuffle=True)
        # 建立 val_loader
        val_loader = create_val_loader(val_df, n_hw,
                                       start_date="2024-01-01", 
                                       end_date="2024-12-20",
                                       batch_size=1)
        
        # 建立模型
        model = TransformerModel().to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
        criterion = nn.MSELoss()
        
        for epoch in range(1, epochs+1):
            # 訓練
            train_loss = train_one_epoch(model, train_loader, optimizer, criterion)
            
            # 驗證，回傳多項指標
            metrics = evaluate_val(model, val_loader)
            
            print(f"  [Epoch {epoch}/{epochs}] TrainLoss={train_loss:.4f} | "
                  f"All_MAE={metrics['mae_all']:.4f}, Tx_MAE={metrics['mae_tx']:.4f}, "
                  f"RH_MAE={metrics['mae_rh']:.4f}, Precp_Acc={metrics['acc_precp']:.4f}")
            
            # 亦可印更多資訊 ...
            
            # 寫 log
            log_entry = {
                "num_history_weeks": n_hw,
                "epoch": epoch,
                "train_loss": train_loss,
                # 加入所有指標
                "mae_all": metrics["mae_all"],
                "mse_all": metrics["mse_all"],
                "rmse_all": metrics["rmse_all"],
                "r2_all": metrics["r2_all"],
                
                "mae_tx": metrics["mae_tx"],
                "mse_tx": metrics["mse_tx"],
                "rmse_tx": metrics["rmse_tx"],
                "r2_tx": metrics["r2_tx"],
                
                "mae_rh": metrics["mae_rh"],
                "mse_rh": metrics["mse_rh"],
                "rmse_rh": metrics["rmse_rh"],
                "r2_rh": metrics["r2_rh"],
                
                "acc_precp": metrics["acc_precp"]
            }
            logs.append(log_entry)
            
            # 即時寫入 log.json
            with open(log_path, "w") as f:
                json.dump(logs, f, indent=4)

In [ ]:
# -----------------------------
# Step 7. 執行實驗
# -----------------------------
if __name__ == "__main__":
    # 你想要嘗試的輸入週數列表
    num_history_weeks_list = [30, 40]  # 可再加大
    
    for i in range(5):
        run_experiment_and_log(
            train_df=train_df,         # 2020~2022 資料
            val_df=val_df_raw,         # 2023+2024 資料
            num_history_weeks_list=num_history_weeks_list,
            epochs=5,
            log_path="log.json"
        )


=== Start Experiment: num_history_weeks=1 ===
  [Epoch 1/5] TrainLoss=435.8570 | All_MAE=1.5942, Tx_MAE=1.6180, RH_MAE=3.0219, Precp_Acc=0.4920
  [Epoch 2/5] TrainLoss=3.9007 | All_MAE=2.3656, Tx_MAE=1.9681, RH_MAE=4.9761, Precp_Acc=0.2746
  [Epoch 3/5] TrainLoss=2.0433 | All_MAE=1.9802, Tx_MAE=1.4499, RH_MAE=4.3455, Precp_Acc=0.3151
  [Epoch 4/5] TrainLoss=1.4645 | All_MAE=1.7752, Tx_MAE=1.4233, RH_MAE=3.7406, Precp_Acc=0.2848
  [Epoch 5/5] TrainLoss=1.1641 | All_MAE=1.5182, Tx_MAE=1.2217, RH_MAE=3.1668, Precp_Acc=0.3568

=== Start Experiment: num_history_weeks=2 ===
  [Epoch 1/5] TrainLoss=441.0860 | All_MAE=1.7807, Tx_MAE=1.9412, RH_MAE=3.2529, Precp_Acc=0.4373
  [Epoch 2/5] TrainLoss=3.5312 | All_MAE=2.3452, Tx_MAE=2.3830, RH_MAE=4.5226, Precp_Acc=0.5961
  [Epoch 3/5] TrainLoss=1.7393 | All_MAE=2.5211, Tx_MAE=1.9260, RH_MAE=5.3905, Precp_Acc=0.2183
  [Epoch 4/5] TrainLoss=1.2533 | All_MAE=2.0234, Tx_MAE=1.4614, RH_MAE=4.3840, Precp_Acc=0.2131
  [Epoch 5/5] TrainLoss=0.9634 | All_M